## Phase 2.1: Data Engineering & Structuring
### Environment & Architecture Setup
To ensure strict reproducibility across the research team's local machines, we establish absolute architectural paths relative to this notebook. This cell initializes the computational environment and guarantees the target directories (`02_intermediate` and `03_processed`) exist before data pipelines are executed.

In [15]:
import pandas as pd
import numpy as np
import os

# 1. Establish strict architectural paths
RAW_DIR = "../data/01_raw/"
INTERMEDIATE_DIR = "../data/02_intermediate/"
PROCESSED_DIR = "../data/03_processed/"

os.makedirs(INTERMEDIATE_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Environment architecture validated.")

Environment architecture validated.


### Comprehensive Data Ingestion
We systematically load all raw data required to test our four core hypotheses. This includes macroeconomic structural variables (World Bank), demand proxies (Google Trends), liquidity and network usage (CoinMetrics, yfinance), and low-level infrastructural network costs (Etherscan, Tronscan).

In [16]:
# 2. Ingest Macro & Crypto Demand Data (H1, H2)
wb_df = pd.read_csv(f"{RAW_DIR}worldbank/all_indicators.csv")
cm_add_df = pd.read_csv(f"{RAW_DIR}coinmetrics/active_addresses.csv")
gt_df = pd.read_csv(f"{RAW_DIR}googletrends/global_search_intensity.csv")
yf_usdc_df = pd.read_csv(f"{RAW_DIR}yfinance/USDC_daily_volume.csv")
yf_usdt_df = pd.read_csv(f"{RAW_DIR}yfinance/USDT_daily_volume.csv")

# 3. Ingest Infrastructure Cost Data (H3, H4)
eth_tx_df = pd.read_csv(f"{RAW_DIR}etherscan/usdc_transfers_sample.csv")
tron_tx_df = pd.read_csv(f"{RAW_DIR}tronscan/usdt_transfers_sample.csv")
eth_price_df = pd.read_csv(f"{RAW_DIR}coinmetrics/eth_trx_price_usd.csv")

print("All raw datasets (Macro, Demand, and Infrastructure) loaded successfully.")

All raw datasets (Macro, Demand, and Infrastructure) loaded successfully.


### Universal Temporal Standardization & Structural Break Injection
Financial APIs and macroeconomic databases format time inconsistently (e.g., Etherscan uses 10-digit UNIX seconds, Tronscan uses 13-digit UNIX milliseconds, World Bank uses annual integers). This function dynamically detects and standardizes all temporal data to a uniform `YYYY-MM-DD` format. 

Furthermore, it strictly truncates all datasets to our defined 2020-2025 timeframe and injects the `post_nov_2022` dummy variable to enable pre/post structural break analysis centered around the FTX liquidity crisis.

In [17]:
def standardize_dates(df, is_annual=False):
    """
    Dynamically identifies temporal columns and robustly handles UNIX seconds 
    (Etherscan) vs UNIX milliseconds (Tronscan) vs standard strings.
    """
    possible_names = ['date', 'Date', 'time', 'timestamp', 'TimeStamp', 'block_timestamp', 'year', 'Year']
    time_col = next((col for col in possible_names if col in df.columns), None)
    
    if not time_col:
        raise KeyError(f"Critical Error: No temporal column found. Available columns: {df.columns}")

    if is_annual:
        df['date'] = pd.to_datetime(df[time_col].astype(str), format='%Y')
        if time_col != 'date':
            df = df.drop(columns=[time_col])
    else:
        # Check if the column is numeric (UNIX timestamps)
        if pd.api.types.is_numeric_dtype(df[time_col]):
            # 10-digit UNIX (Seconds - e.g., Etherscan)
            if df[time_col].max() < 20000000000: 
                df['date'] = pd.to_datetime(df[time_col], unit='s', errors='coerce')
            # 13-digit UNIX (Milliseconds - e.g., Tronscan)
            else:
                df['date'] = pd.to_datetime(df[time_col], unit='ms', errors='coerce')
        else:
            # Standard string dates
            df['date'] = pd.to_datetime(df[time_col], errors='coerce')

    if df['date'].dt.tz is not None:
        df['date'] = df['date'].dt.tz_localize(None)
    df['date'] = df['date'].dt.normalize()
    
    # 2020-2025 Truncation & Structural Break Dummy
    mask = (df['date'] >= '2020-01-01') & (df['date'] <= '2025-12-31')
    df = df.loc[mask].copy()
    df['post_nov_2022'] = (df['date'] > '2022-11-30').astype(int)
    
    if time_col != 'date' and time_col in df.columns:
         df = df.drop(columns=[time_col])
            
    return df.sort_values(by=['date']).reset_index(drop=True)

# Apply bulletproof standardizer
wb_df = standardize_dates(wb_df, is_annual=True)
cm_add_df = standardize_dates(cm_add_df)
gt_df = standardize_dates(gt_df)
yf_usdc_df = standardize_dates(yf_usdc_df)
yf_usdt_df = standardize_dates(yf_usdt_df)
eth_tx_df = standardize_dates(eth_tx_df)
tron_tx_df = standardize_dates(tron_tx_df)
eth_price_df = standardize_dates(eth_price_df)

print("Temporal standardization and 'post_nov_2022' dummy injected across ALL datasets.")

Temporal standardization and 'post_nov_2022' dummy injected across ALL datasets.


### Macroeconomic Imputation & The "Namibia" Correction
Macroeconomic variables such as inflation and banking penetration inherently suffer from reporting lags. Dropping these rows introduces severe truncation bias. 

First, we resolve a known pandas ingestion anomaly where Namibia's ISO code ('NA') is misread as a Null value. We then apply strict linear interpolation, grouped by sovereign country codes, to mathematically estimate the 17 missing macro reporting gaps without bleeding data across borders.

In [18]:
# Fix Pandas "Namibia" bug ('NA' read as Null)
if 'country_name' in wb_df.columns:
    wb_df.loc[wb_df['country_name'] == 'Namibia', 'country_code'] = 'NAM'
    wb_df['country_code'] = wb_df.groupby('country_name')['country_code'].transform(lambda x: x.ffill().bfill())
    wb_df['country_code'] = wb_df['country_code'].fillna('UNKNOWN')

# Isolate numeric columns for strict mathematical interpolation
numeric_cols = [c for c in wb_df.select_dtypes(include=[np.number]).columns if c not in ['post_nov_2022']]

# Interpolate missing trends
if 'country_code' in wb_df.columns:
    wb_df[numeric_cols] = wb_df.groupby('country_code')[numeric_cols].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
wb_df[numeric_cols] = wb_df[numeric_cols].ffill().bfill()

assert wb_df.isna().sum().sum() == 0, "Nulls persist in Macro data."
print("World Bank macroeconomic data fully imputed and validated.")

World Bank macroeconomic data fully imputed and validated.


### Infrastructure Outlier Filtering & USD Normalization (Hypothesis 4)
To accurately compare the empirical "cost friction" of blockchain networks against legacy correspondent banking (SWIFT), we must normalize raw protocol fees (Gwei) into USD using historical daily prices. 

Crucially, we filter out the top 5% of Ethereum transaction fees. This isolates the baseline utility cost of stablecoin transfers by removing extreme outliers caused by speculative network congestion (e.g., NFT mint gas wars), which would otherwise artificially skew our H4 cost regressions.

In [19]:
# 1. USD Normalization: Merge daily ETH price to calculate exact USD cost
eth_tx_df = eth_tx_df.merge(eth_price_df[['date', 'PriceUSD']], on='date', how='left')

# Find the gas fee column dynamically (handles 'gas_fee_eth', 'fee', 'gas_fee')
gas_col = next((col for col in eth_tx_df.columns if 'fee' in col.lower()), None)

if gas_col and 'PriceUSD' in eth_tx_df.columns:
    eth_tx_df['fee_usd'] = eth_tx_df[gas_col] * eth_tx_df['PriceUSD']
    
    # 2. Outlier Filtering: Remove top 5% gas spikes (NFT mint congestion)
    initial_count = len(eth_tx_df)
    gas_cap_95th = eth_tx_df['fee_usd'].quantile(0.95)
    eth_tx_df = eth_tx_df[eth_tx_df['fee_usd'] <= gas_cap_95th].copy()
    
    print(f"Filtered {initial_count - len(eth_tx_df)} extreme ETH gas outliers (> ${gas_cap_95th:.2f}).")
else:
    print("Warning: Could not dynamically calculate fee_usd. Check Etherscan column structures.")

Filtered 7200 extreme ETH gas outliers (> $47.45).


### Master Export to Intermediate Directory
With all temporal misalignments, null values, and infrastructural outliers completely resolved, we export these 7 atomic datasets to the `02_intermediate` directory. This concludes Phase 2.1.

In [20]:
# Save all strictly clean, atomic datasets
wb_df.to_csv(f"{INTERMEDIATE_DIR}wb_cleaned.csv", index=False)
cm_add_df.to_csv(f"{INTERMEDIATE_DIR}cm_active_addresses_cleaned.csv", index=False)
gt_df.to_csv(f"{INTERMEDIATE_DIR}gt_global_intensity_cleaned.csv", index=False)
yf_usdc_df.to_csv(f"{INTERMEDIATE_DIR}yf_usdc_volume_cleaned.csv", index=False)
yf_usdt_df.to_csv(f"{INTERMEDIATE_DIR}yf_usdt_volume_cleaned.csv", index=False)
eth_tx_df.to_csv(f"{INTERMEDIATE_DIR}etherscan_usdc_cleaned.csv", index=False)
tron_tx_df.to_csv(f"{INTERMEDIATE_DIR}tronscan_usdt_cleaned.csv", index=False)

print("PHASE 2.1 OFFICIALLY COMPLETE. All 7 datasets safely exported to /02_intermediate/")

PHASE 2.1 OFFICIALLY COMPLETE. All 7 datasets safely exported to /02_intermediate/


### Phase 2.2, Task 1: Structuring the Hypothesis 1 (H1) Master Dataset

**Economic Rationale:** To empirically test Hypothesis 1 (Network Effects), we must construct a dataset that isolates the relationship between network utility (transfer volume) and user adoption (active addresses). According to Metcalfe's Law, the value of a network is proportional to the square of its users. 

This cell ingests the distinctly exported volume and active address datasets from Phase 2.1, standardizes their naming conventions, and merges them into a unified panel format. We compute the square of active addresses to establish the quantitative foundation necessary for our Phase 3 OLS regression, allowing us to observe if stablecoins exhibit the runaway liquidity required to challenge SWIFT.

In [22]:
import pandas as pd
import os

# 1. Ensure Autonomous Reproducibility via Relative Paths
INPUT_DIR = '../data/02_intermediate/'
OUTPUT_DIR = '../data/03_processed/'

# 2. Ingest intermediate datasets (without relying on default parse_dates)
cm_addr_df = pd.read_csv(f'{INPUT_DIR}cm_active_addresses_cleaned.csv')
usdc_vol_df = pd.read_csv(f'{INPUT_DIR}yf_usdc_volume_cleaned.csv')
usdt_vol_df = pd.read_csv(f'{INPUT_DIR}yf_usdt_volume_cleaned.csv')

# 3. Defensively Force Date Formats 
# This completely strips out any lingering timezones or hour/minute variations
for df in [cm_addr_df, usdc_vol_df, usdt_vol_df]:
    df['date'] = pd.to_datetime(df['date'], utc=True).dt.tz_localize(None).dt.normalize()

# 4. Defensively Standardize CoinMetrics Asset Tickers
# CoinMetrics often returns assets like 'usdt', 'usdt_eth', or 'usdt_trx'.
def map_stablecoin_ticker(ticker):
    ticker = str(ticker).upper()
    if 'USDC' in ticker: return 'USDC'
    if 'USDT' in ticker: return 'USDT'
    return 'OTHER'

cm_addr_df['asset'] = cm_addr_df['asset'].apply(map_stablecoin_ticker)

# Filter out anything that isn't our two target stablecoins
cm_addr_df = cm_addr_df[cm_addr_df['asset'].isin(['USDC', 'USDT'])]

# If CoinMetrics split active addresses by chain (e.g., USDT on Tron + USDT on Eth),
# we must aggregate them by date and asset to avoid duplicate key explosions or missed joins.
cm_addr_grouped = cm_addr_df.groupby(['date', 'asset'], as_index=False)['AdrActCnt'].sum()

# 5. Prepare Yahoo Finance Volume Data
usdc_vol_df['asset'] = 'USDC'
usdt_vol_df['asset'] = 'USDT'
combined_vol_df = pd.concat([usdc_vol_df, usdt_vol_df], ignore_index=True)

# 6. Execute the Merge (The Composite Key intersection should now be flawless)
h1_processed = pd.merge(
    combined_vol_df[['date', 'asset', 'volume_usd', 'post_nov_2022']], 
    cm_addr_grouped[['date', 'asset', 'AdrActCnt']], 
    on=['date', 'asset'], 
    how='inner'
)

# Rename columns for econometric clarity
h1_processed = h1_processed.rename(columns={
    'volume_usd': 'transfer_volume_usd',
    'AdrActCnt': 'active_addresses'
})

# 7. Feature Engineering: Metcalfe's Law
h1_processed['active_addresses_squared'] = h1_processed['active_addresses'] ** 2

# 8. Final Panel Sorting and Export
h1_processed = h1_processed.sort_values(by=['asset', 'date']).reset_index(drop=True)
h1_processed.to_csv(f'{OUTPUT_DIR}h1_network_effects.csv', index=False)

# 9. Validation Diagnostics
print(f"H1 Processed Panel Shape: {h1_processed.shape}")
print("\nMissing Values:")
print(h1_processed.isna().sum())
print("\nFirst 3 rows per asset:")
display(h1_processed.groupby('asset').head(3))

H1 Processed Panel Shape: (4382, 6)

Missing Values:
date                        0
asset                       0
transfer_volume_usd         0
post_nov_2022               0
active_addresses            0
active_addresses_squared    0
dtype: int64

First 3 rows per asset:


,date,asset,transfer_volume_usd,post_nov_2022,active_addresses,active_addresses_squared
0,2020-01-01,USDC,242586528,0,1481,2193361
1,2020-01-02,USDC,318268134,0,2302,5299204
2,2020-01-03,USDC,374792167,0,2561,6558721
2191,2020-01-01,USDT,21503143454,0,28037,786073369
2192,2020-01-02,USDT,24212314977,0,32087,1029575569
2193,2020-01-03,USDT,32420287856,0,49281,2428616961


## Phase 2.2 - Task 2: Structuring the H2 Diffusion Dataset
**Economic Rationale:** Hypothesis 2 posits that stablecoin adoption accelerates in regions suffering from legacy banking and institutional inefficiencies. To test this empirically via a cross-country panel regression, we must construct a Master Diffusion Dataset. This requires transforming the World Bank structural indicators (longitudinal panel data) into distinct feature columns and integrating them with stablecoin demand proxies (Google Trends search intensity). The inclusion of the `post_nov_2022` dummy variable ensures we can test for structural breaks in adoption elasticity following the late-2022 crypto liquidity crises.

In [29]:
import pandas as pd
import numpy as np

# Define input paths
WB_FILE = "../data/02_intermediate/wb_cleaned.csv"
GT_FILE = "../data/02_intermediate/gt_global_intensity_cleaned.csv"
PROCESSED_DIR = "../data/03_processed/"

# 1. Ingest Data
df_wb = pd.read_csv(WB_FILE, parse_dates=['date'])
df_gt = pd.read_csv(GT_FILE, parse_dates=['date'])

# 2. Pivot World Bank Panel Data
# We transition from a long format (indicators in rows) to a wide format (indicators as columns)
df_wb_pivoted = df_wb.pivot_table(
    index=['country_code', 'country_name', 'date', 'post_nov_2022'],
    columns='indicator_name',
    values='value',
    aggfunc='mean'
).reset_index()

# Clean column names to remove pivot artifacts
df_wb_pivoted.columns.name = None

# 3. Merge Panel with Google Trends Demand Proxy
# Using a left join to map the global demand index to each country's temporal observation
h2_diffusion = pd.merge(
    df_wb_pivoted,
    df_gt,
    on=['date', 'post_nov_2022'],
    how='left'
)

# 4. Sort Panel for Econometric Rigor (Country, then Time)
h2_diffusion = h2_diffusion.sort_values(by=['country_code', 'date']).reset_index(drop=True)

# 5. Export Master Dataset
h2_diffusion.to_csv(f"{PROCESSED_DIR}h2_diffusion_dataset.csv", index=False)

# 6. Validation Diagnostics
print(f"H2 Master Dataset Shape: {h2_diffusion.shape}")
print("\nMissing Values per Feature:")
print(h2_diffusion.isna().sum())
print("\nPanel Data Sample (First 5 Rows):")
display(h2_diffusion.head(5))

H2 Master Dataset Shape: (1276, 11)

Missing Values per Feature:
country_code                         0
country_name                         0
date                                 0
post_nov_2022                        0
financial_account_ownership_pct    965
gdp_per_capita_usd                  12
inflation_cpi_annual_pct           155
remittances_received_pct_gdp       128
USDT                                 0
USDC                                 0
stablecoin                           0
dtype: int64

Panel Data Sample (First 5 Rows):


,country_code,country_name,date,post_nov_2022,financial_account_ownership_pct,gdp_per_capita_usd,inflation_cpi_annual_pct,remittances_received_pct_gdp,USDT,USDC,stablecoin
0,1A,Arab World,2020-01-01,0,NaN,5739.412836,1.447037,2.777971,1,0,0
1,1A,Arab World,2021-01-01,0,40.21,6697.414985,2.868330,2.694799,6,1,1
2,1A,Arab World,2022-01-01,0,NaN,7950.355820,5.179808,2.142007,24,3,1
3,1A,Arab World,2023-01-01,1,NaN,7513.899079,4.000259,1.685865,16,1,0
4,1A,Arab World,2024-01-01,1,NaN,7583.811701,2.101356,1.681060,22,2,0


## Phase 2.2 - Task 3: Structuring the H4 Cost Friction Dataset
**Economic Rationale:** Hypothesis 4 states that stablecoins replace legacy correspondent banking by substituting massive fixed infrastructural costs with variable network fees (gas). To test the "Cost Friction Spread," we must aggregate transaction-level blockchain data (millions of rows from Ethereum and Tron) into a clean time-series panel of daily average transaction fees. This dataset will directly compare the frictional cost of moving USDC via Ethereum versus USDT via Tron, enabling us to measure the protocol efficiency gain against legacy SWIFT baseline costs.

In [31]:
import pandas as pd
import numpy as np

# Define input paths
ETH_FILE = "../data/02_intermediate/etherscan_usdc_cleaned.csv"
TRX_FILE = "../data/02_intermediate/tronscan_usdt_cleaned.csv"
PROCESSED_DIR = "../data/03_processed/"

# 1. Ingest Transaction-Level Data
df_eth = pd.read_csv(ETH_FILE)
df_trx = pd.read_csv(TRX_FILE)

# 2. Aggregate Ethereum (USDC) MONTHLY Averages
# Grouping by 'month' fixes the date-staggering anomaly from the raw scrapers
eth_monthly = df_eth.groupby(['month', 'post_nov_2022']).agg(
    eth_mean_fee_usd=('fee_usd', 'mean'),
    eth_mean_value_usdc=('value_usdc', 'mean'),
    eth_tx_count=('tx_hash', 'count')
).reset_index()

# 3. Aggregate Tron (USDT) MONTHLY Averages
trx_monthly = df_trx.groupby(['month', 'post_nov_2022']).agg(
    trx_mean_fee_trx=('fee_trx', 'mean'),
    trx_mean_value_usdt=('value_usdt', 'mean'),
    trx_tx_count=('tx_hash', 'count')
).reset_index()

# 4. Merge into a Master Cost Panel on the Month vector
h4_cost_friction = pd.merge(
    eth_monthly,
    trx_monthly,
    on=['month', 'post_nov_2022'],
    how='inner'  # Inner join perfectly aligns the months without NaNs
)

# 5. Integrate the Legacy Correspondent Banking Baseline
# Applying the World Bank global average remittance friction (6.20%)
h4_cost_friction['legacy_fee_pct_baseline'] = 0.0620

# Calculate the implied fixed cost if the Ethereum USDC transfer value was sent via SWIFT/Remittance
h4_cost_friction['legacy_implied_fee_usd'] = h4_cost_friction['eth_mean_value_usdc'] * h4_cost_friction['legacy_fee_pct_baseline']

# Calculate the "Cost Friction Spread" (Legacy Fee - Crypto Fee)
# This explicitly tests Hypothesis 4: The cost savings of protocol vs legacy infrastructure
h4_cost_friction['cost_savings_usd_eth'] = h4_cost_friction['legacy_implied_fee_usd'] - h4_cost_friction['eth_mean_fee_usd']

# 6. Sort Time-Series and Export
h4_cost_friction = h4_cost_friction.sort_values(by='month').reset_index(drop=True)
h4_cost_friction.to_csv(f"{PROCESSED_DIR}h4_infrastructure_cost.csv", index=False)

# 7. Validation Diagnostics
print(f"H4 Cost Dataset Shape: {h4_cost_friction.shape}")
print("\nMissing Values per Feature:")
print(h4_cost_friction.isna().sum())
print("\nCost Friction Sample (First 3 Rows):")
display(h4_cost_friction.head(3))

H4 Cost Dataset Shape: (72, 11)

Missing Values per Feature:
month                      0
post_nov_2022              0
eth_mean_fee_usd           0
eth_mean_value_usdc        0
eth_tx_count               0
trx_mean_fee_trx           0
trx_mean_value_usdt        0
trx_tx_count               0
legacy_fee_pct_baseline    0
legacy_implied_fee_usd     0
cost_savings_usd_eth       0
dtype: int64

Cost Friction Sample (First 3 Rows):


,month,post_nov_2022,eth_mean_fee_usd,eth_mean_value_usdc,eth_tx_count,trx_mean_fee_trx,trx_mean_value_usdt,trx_tx_count,legacy_fee_pct_baseline,legacy_implied_fee_usd,cost_savings_usd_eth
0,2020-01,0,0.081304,19155.262157,2000,0.043928,10291.878332,20,0.062,1187.626254,1187.544950
1,2020-02,0,0.103677,15540.955159,2000,0.105382,14817.418934,20,0.062,963.539220,963.435543
2,2020-03,0,0.197608,29601.378386,2000,0.066438,14010.455311,20,0.062,1835.285460,1835.087852
